# Missing-view survival prediction (simulation)

Fit discrete-time logistic hazards across six intervals, with and without duplicating training records after masking the second feature group. Test on complete and masked inputs. Score one-year predicted risk using Harrell concordance across all comparable follow-up pairs. Each seed creates 2,600 records with independent exponential censoring and event times influenced by both feature groups. The holdout is 30%.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'survival-view-dropout-heavy-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", str(ROOT / "data")))\nSEEDS = [13, 37, 61]\n\n\ndef save(model, name):\n    path = ROOT / "checkpoints"\n    path.mkdir(exist_ok=True)\n    joblib.dump(model, path / (name + ".joblib"), compress=3)\n\n\nfrom sklearn.impute import SimpleImputer\n\n\ndef design(x, stop, event):\n    cuts = np.array([90, 180, 270, 365, 550, 730])\n    rows = []\n    labels = []\n    for start, end in zip([0, *cuts[:-1]], cuts):\n        use = (stop > start) & ((stop >= end) | event)\n        rows.extend(np.column_stack([x[use], np.full(use.sum(), np.log1p(start))]))\n        labels.extend(event[use] & (stop[use] <= end))\n    return np.array(rows), np.array(labels)\n\n\ndef run(seed):\n    rng = np.random.default_rng(seed)\n    x = rng.normal(size=(2600, 7))\n    event = rng.exponential(450 * np.exp(-0.55 * x[:, 0] - 0.8 * x[:, 4]))\n    cens = rng.exponential(750, size=len(x))\n    t = np.minimum(event, cens)\n    e = event <= cens\n    tr, te = train_test_split(\n        np.arange(len(x)), test_size=0.30, stratify=e, random_state=seed\n    )\n\n    def masked(a):\n        a = a.copy()\n        a[:, 4:] = np.nan\n        return a\n\n    out = {}\n    for name in ["complete_training", "view_dropout"]:\n        a = x[tr]\n        times = t[tr]\n        events = e[tr]\n        if name == "view_dropout":\n            a = np.vstack([a, masked(a)])\n            times = np.tile(times, 2)\n            events = np.tile(events, 2)\n        features, target = design(a, times, events)\n        model = make_pipeline(\n            SimpleImputer(add_indicator=True),\n            StandardScaler(),\n            LogisticRegression(C=0.15, max_iter=500),\n        ).fit(features, target)\n        save(model, f"{name}-{seed}")\n        for condition, z in [("available", x[te]), ("missing", masked(x[te]))]:\n            hazard = np.array(\n                [\n                    model.predict_proba(np.c_[z, np.full(len(z), np.log1p(b))])[:, 1]\n                    for b in [0, 90, 180, 270]\n                ]\n            )\n            risk = 1 - (1 - hazard).prod(0)\n            comparable = (t[te, None] < t[te][None, :]) & e[te, None]\n            diff = risk[:, None] - risk\n            out[name + "_" + condition] = {\n                "concordance": float(\n                    ((diff > 0) + 0.5 * (diff == 0))[comparable].mean()\n                ),\n                "comparable_pairs": int(comparable.sum()),\n            }\n    return out\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())